# Kerala oral cancer screening trial: participant flow and mortality rate ratios

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shah-pushti/oral-cancer-figures/blob/main/notebooks/kerala-trial.ipynb)

This notebook rebuilds the figures of the article [Oral visual screening and deaths from oral cancer: results of the Kerala cluster-randomised trial](https://shah-pushti.github.io/oral-cancer-screening-kerala-trial.html). The trial randomised 13 clusters in Trivandrum district, Kerala, India, to three rounds of oral visual screening or to no screening rounds (1996 to 2004).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Read the data from this repository; in Colab, read it from GitHub.
BASE = Path("..") if Path("../data").exists() else None
RAW = "https://raw.githubusercontent.com/shah-pushti/oral-cancer-figures/main/"

def load(name):
    return pd.read_csv(BASE / "data" / name if BASE else RAW + "data/" + name)

INK, BURGUNDY, STEEL, PAPER = "#231920", "#6E1A2D", "#3F6068", "#F7F8F5"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
})
OUT = Path("../figures") if BASE else Path("figures")
OUT.mkdir(exist_ok=True)

In [ ]:
flow = load("kerala-trial-flow.csv")
flow

The counts after referral add up: people who attended referral equal the sum of the three diagnostic outcomes, and screen-positive people equal those who attended plus those who did not.

In [ ]:
c = flow.set_index("stage")["count"]
assert c["attended referral"] == c["oral precancerous lesions diagnosed"] + c["invasive oral cancer diagnosed"] + c["benign lesions or normal findings"]
assert c["screen-positive"] == c["attended referral"] + c["did not attend referral"]
print(f"Attended referral: {c['attended referral'] / c['screen-positive']:.0%} of screen-positive people")

In [ ]:
rr = load("kerala-trial-mortality-rate-ratios.csv")
rr[["group", "rate_ratio", "ci95_low", "ci95_high"]]

The rate ratio divides the oral cancer mortality rate in the intervention clusters by the rate in the control clusters. Ratios are plotted on a logarithmic axis, so that 0.5 and 2 lie at equal distances from 1. A filled dot marks an interval that excludes 1; an open dot marks an interval that includes 1.

In [ ]:
import numpy as np
fig, ax = plt.subplots(figsize=(8, 3.2))
for y, r in enumerate(rr[::-1].itertuples()):
    excl = r.ci95_high < 1 or r.ci95_low > 1
    ax.plot([r.ci95_low, r.ci95_high], [y, y], color=BURGUNDY, linewidth=2)
    ax.plot(r.rate_ratio, y, "o", markersize=8, color=BURGUNDY, markerfacecolor=BURGUNDY if excl else PAPER)
    ax.text(1.75, y, f"{r.rate_ratio:.2f} ({r.ci95_low:.2f} to {r.ci95_high:.2f})", va="center", fontsize=9)
ax.axvline(1, color=INK, linestyle="--", linewidth=0.8)
ax.set_xscale("log"); ax.xaxis.set_minor_formatter(plt.NullFormatter()); ax.set_xticks([0.3, 0.5, 1, 1.5]); ax.set_xticklabels(["0.3", "0.5", "1", "1.5"]); ax.set_xticks([], minor=True)
ax.set_xlim(0.3, 2.6)
ax.set_yticks(range(len(rr))); ax.set_yticklabels(rr["group"][::-1])
ax.set_xlabel("Oral cancer mortality rate ratio, intervention versus control (95% CI)")
fig.text(0.01, -0.04, "Source: Sankaranarayanan R, et al. The Lancet 2005;365:1927-33.", fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "kerala-rate-ratios.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT / "kerala-rate-ratios.svg", bbox_inches="tight")

## Sources

- Sankaranarayanan R, Ramadas K, Thomas G, et al. Effect of screening on oral cancer mortality in Kerala, India: a cluster-randomised controlled trial. The Lancet 2005;365:1927-33. https://doi.org/10.1016/S0140-6736(05)66658-5